# OOF-анализ: одна строка на пару

Собирает единую таблицу, где для каждой из 365 654 размеченных пар лежат:

* метка, категория и номер фолда,
* out-of-fold предсказание **каждого** зарегистрированного эксперимента,
* тексты обоих товаров — название и атрибуты.

OOF означает, что предсказание для пары сделано моделью, которая эту пару
не видела при обучении: фолды сгруппированы по компонентам связности, поэтому
ни один товар из фолда не встречается в его обучающих парах. Это делает
таблицу честной основой для разбора ошибок — в отличие от предсказаний на
обучающей выборке.

Ничего не обучает и ничего не пишет на диск; только читает готовые артефакты.

In [1]:
from pathlib import Path

import numpy as np
import polars as pl

# работает и из каталога ноутбука, и из корня репозитория
REPO = Path.cwd().resolve()
while not (REPO / "validation" / "folds.json").is_file():
    if REPO.parent == REPO:
        raise SystemExit("репозиторий не найден — запусти ноутбук внутри него")
    REPO = REPO.parent

SPEC = "v2"  # "v1" — исходные хешированные фолды, "v2" — стратифицированные
suffix = "" if SPEC == "v1" else f"_{SPEC}"
TARGETS = REPO / "validation" / f"targets{suffix}"
PREDICTIONS = REPO / "validation" / f"predictions{suffix}" / "darksteeld"
RESULTS = REPO / "validation" / f"results{suffix}" / "darksteeld"
RAW = REPO / "data" / "raw"
FOLDS = [f"fold_{k:02d}" for k in range(1, 5)]

print(f"репозиторий {REPO}")
for path in (TARGETS, PREDICTIONS, RAW):
    print(f"  {'есть ' if path.is_dir() else 'НЕТ  '} {path.relative_to(REPO)}")
if not TARGETS.is_dir():
    print("\nцелей нет — собери их: make validation-targets" + ("-v2" if SPEC == "v2" else ""))

репозиторий /Users/dmitrijrudenko/Documents/dev/ozon1
  есть  validation/targets_v2
  есть  validation/predictions_v2/darksteeld
  есть  data/raw


## 1. Метки фолдов

Файлы целей локальны и не хранятся в Git; порядок пар в них канонический
(сортировка по `id1, id2`), и предсказания обязаны ему следовать.

In [2]:
targets = pl.concat(
    [pl.read_csv(TARGETS / f"{f}.csv").with_columns(pl.lit(f).alias("fold")) for f in FOLDS]
)
print(f"{targets.height:,} пар, доля позитивов {targets['target'].mean():.4f}")
print(targets.group_by("fold").agg(
    pl.len().alias("пар"), pl.col("target").mean().round(4).alias("pos_rate")
).sort("fold"))
targets.head(3)

365,654 пар, доля позитивов 0.2568
shape: (4, 3)
┌─────────┬───────┬──────────┐
│ fold    ┆ пар   ┆ pos_rate │
│ ---     ┆ ---   ┆ ---      │
│ str     ┆ u32   ┆ f64      │
╞═════════╪═══════╪══════════╡
│ fold_01 ┆ 91408 ┆ 0.2568   │
│ fold_02 ┆ 91414 ┆ 0.2568   │
│ fold_03 ┆ 91416 ┆ 0.2568   │
│ fold_04 ┆ 91416 ┆ 0.2567   │
└─────────┴───────┴──────────┘


id1,id2,target,category,fold
i64,i64,i64,str,str
192,257698056754,0,"""Красота и гигиена""","""fold_01"""
476,841813632218,1,"""Бытовая техника""","""fold_01"""
700,798863923139,0,"""Строительство и ремонт""","""fold_01"""


## 2. OOF-предсказания всех экспериментов

Каталоги находятся автоматически. Совпадение порядка пар проверяется, а не
предполагается: если эксперимент писал предсказания в другом порядке, тихая
рассинхронизация испортила бы весь дальнейший анализ.

In [3]:
models = sorted(
    d.name for d in PREDICTIONS.iterdir()
    if d.is_dir() and all((d / f"{f}.csv").is_file() for f in FOLDS)
)

oof = targets.clone()
for model in models:
    frame = pl.concat([pl.read_csv(PREDICTIONS / model / f"{f}.csv") for f in FOLDS])
    assert frame.height == oof.height, f"{model}: {frame.height} строк против {oof.height}"
    assert frame["id1"].equals(oof["id1"]) and frame["id2"].equals(oof["id2"]), (
        f"{model}: порядок пар отличается от целей"
    )
    oof = oof.with_columns(frame["predict"].alias(model))

print(f"{len(models)} экспериментов, порядок пар совпадает у всех:")
for model in models:
    print(f"  {model}")

25 экспериментов, порядок пар совпадает у всех:
  attr_jaccard
  blend3_equal
  blend3_opt
  blend_lgbm_knrm_50
  blend_lgbm_knrm_audit_50
  const_prior
  knrm_attrs
  knrm_attrs_llm
  knrm_attrs_llm_full
  knrm_joint
  knrm_llm_audit
  knrm_llm_pretrain
  knrm_name
  knrm_name_noaudit
  knrm_name_v2
  lgbm_cheap_audit
  lgbm_cheap_v1
  lgbm_knrm_insample
  lgbm_knrm_nested
  lgbm_llm
  name_exact
  name_tfidf_attr_blend
  name_tfidf_cos
  stack3_logit
  stack3_logit_full


## 3. Сырые данные товаров

`items_human.parquet` покрывает все размеченные пары, поэтому join не теряет
строк. Берём **все сырые колонки обеих сторон как есть** — атрибуты остаются
строкой JSON из исходного parquet, без разбора.

`category1` и `category2` оставлены, хотя и дублируют `category` пары:
их равенство проверяется ассертом, а не принимается на веру.

In [4]:
items = pl.read_parquet(RAW / "items_human.parquet")
print(f"items_human: {items.height:,} товаров, колонки {items.columns}")

# все колонки обеих сторон, ничего не отбрасываем
left = items.rename({c: f"{c}1" for c in items.columns})
right = items.rename({c: f"{c}2" for c in items.columns})
df = oof.join(left, on="id1", how="left").join(right, on="id2", how="left")

assert df.height == oof.height, "join размножил строки"
for column in ("name1", "name2", "attributes1", "attributes2"):
    assert df[column].null_count() == 0, f"{column}: есть пропуски"
assert (df["category1"] == df["category"]).all() and (df["category2"] == df["category"]).all(), (
    "категория стороны разошлась с категорией пары"
)

print(f"\nитог: {df.height:,} строк x {df.width} колонок")
for column in df.columns:
    print(f"  {column:<24} {df.schema[column]}")

length = df["attributes1"].str.len_chars()
print(f"\natributes лежат строкой JSON как в исходном parquet; длина в символах: "
      f"медиана {int(length.median())}, p95 {int(length.quantile(0.95))}, максимум {length.max():,}")

# для тех, кому привычнее pandas:  pdf = df.to_pandas()

items_human: 711,304 товаров, колонки ['id', 'name', 'attributes', 'category']

итог: 365,654 строк x 36 колонок
  id1                      Int64
  id2                      Int64
  target                   Int64
  category                 String
  fold                     String
  attr_jaccard             Float64
  blend3_equal             Float64
  blend3_opt               Float64
  blend_lgbm_knrm_50       Float64
  blend_lgbm_knrm_audit_50 Float64
  const_prior              Float64
  knrm_attrs               Float64
  knrm_attrs_llm           Float64
  knrm_attrs_llm_full      Float64
  knrm_joint               Float64
  knrm_llm_audit           Float64
  knrm_llm_pretrain        Float64
  knrm_name                Float64
  knrm_name_noaudit        Float64
  knrm_name_v2             Float64
  lgbm_cheap_audit         Float64
  lgbm_cheap_v1            Float64
  lgbm_knrm_insample       Float64
  lgbm_knrm_nested         Float64
  lgbm_llm                 Float64
  name_exact        

Атрибуты бывают в тысячи символов, а polars по умолчанию обрезает строки при
выводе. Ниже — настройка показа и хелпер, печатающий пару целиком.

In [5]:
pl.Config.set_fmt_str_lengths(160)     # сколько символов строки показывать в таблицах
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(220)


def show_pair(row, attribute_chars: int | None = None) -> None:
    """Одна пара целиком: метка, скоры всех моделей и сырые тексты без обрезки.

    row — номер строки в df или уже готовый dict из iter_rows(named=True).
    attribute_chars ограничивает вывод атрибутов, если он слишком длинный.
    """
    r = df.row(row, named=True) if isinstance(row, int) else row
    print(f"target={r['target']}  [{r['category']}]  {r['fold']}  id {r['id1']} <-> {r['id2']}")
    print("  " + "  ".join(f"{m}={r[m]:.4f}" for m in models))
    for side in ("1", "2"):
        attributes = r[f"attributes{side}"]
        if attribute_chars is not None and len(attributes) > attribute_chars:
            attributes = attributes[:attribute_chars] + f"… (+{len(attributes) - attribute_chars} симв.)"
        print(f"\n  --- сторона {side}   id {r['id' + side]}")
        print(f"  name       : {r['name' + side]}")
        print(f"  attributes : {attributes}")


show_pair(1)

target=1  [Бытовая техника]  fold_01  id 476 <-> 841813632218
  attr_jaccard=0.0244  blend3_equal=0.4013  blend3_opt=0.4069  blend_lgbm_knrm_50=0.4295  blend_lgbm_knrm_audit_50=0.4249  const_prior=0.2568  knrm_attrs=0.3146  knrm_attrs_llm=0.3447  knrm_attrs_llm_full=0.4204  knrm_joint=0.3303  knrm_llm_audit=0.4329  knrm_llm_pretrain=0.4353  knrm_name=0.4267  knrm_name_noaudit=0.3270  knrm_name_v2=0.2622  lgbm_cheap_audit=0.4168  lgbm_cheap_v1=0.4238  lgbm_knrm_insample=0.2622  lgbm_knrm_nested=0.4000  lgbm_llm=0.4654  name_exact=0.0000  name_tfidf_attr_blend=0.2998  name_tfidf_cos=0.5752  stack3_logit=0.4646  stack3_logit_full=0.5019

  --- сторона 1   id 476
  name       : микроволновая печь соло lg ms-20r42d
  attributes : {"бренд":"lg","цвет товара":"белый","программы приготовления":"разморозка","страна-изготовитель":"китай","глубина, см":"31.2","диаметр поворотного стола, см":"24.5","размеры, мм":"455 x 284 x 312","особенности микроволновой печи":"быстрый старт","высота, см":"28.4"

## 4. Самопроверка

Пересчитываем PR-AUC из загруженной таблицы и сверяем с зарегистрированными
результатами. Если что-то разошлось — таблица собрана не из тех файлов, и
дальше идти нельзя.

In [6]:
import json


def average_precision(target: np.ndarray, score: np.ndarray) -> float:
    """Та же формула, что в validation/evaluate.py, включая обработку ties."""
    order = np.argsort(-score, kind="mergesort")
    labels, ranked = target[order], score[order]
    cumulative = np.cumsum(labels)
    if cumulative[-1] == 0:
        return float("nan")
    last = np.r_[ranked[1:] != ranked[:-1], True]
    precision = cumulative[last] / (np.arange(len(labels))[last] + 1)
    recall = cumulative[last] / cumulative[-1]
    return float(np.sum(np.diff(np.r_[0.0, recall]) * precision))


target = df["target"].to_numpy().astype(float)
print(f"{'эксперимент':<24}{'mean OOF':>10}{'в результатах':>15}{'расхождение':>13}")
for model in models:
    score = df[model].to_numpy()
    per_fold = [
        average_precision(target[(df['fold'] == f).to_numpy()], score[(df['fold'] == f).to_numpy()])
        for f in FOLDS
    ]
    mean = float(np.mean(per_fold))
    path = RESULTS / f"{model}.json"
    registered = json.loads(path.read_text())["mean_prauc"] if path.is_file() else float("nan")
    flag = "" if abs(mean - registered) < 1e-9 else "  <-- РАСХОЖДЕНИЕ"
    print(f"{model:<24}{mean:10.5f}{registered:15.5f}{mean - registered:+13.2e}{flag}")

эксперимент               mean OOF  в результатах  расхождение
attr_jaccard               0.25469        0.25469    +0.00e+00
blend3_equal               0.67416        0.67416    +0.00e+00
blend3_opt                 0.68110        0.68110    +0.00e+00
blend_lgbm_knrm_50         0.66312        0.66312    +0.00e+00
blend_lgbm_knrm_audit_50   0.66324        0.66324    +0.00e+00
const_prior                0.25677        0.25677    +0.00e+00
knrm_attrs                 0.55188        0.55188    +0.00e+00
knrm_attrs_llm             0.57075        0.57075    +0.00e+00
knrm_attrs_llm_full        0.57034        0.57034    +0.00e+00
knrm_joint                 0.61915        0.61915    +0.00e+00
knrm_llm_audit             0.56571        0.56571    +0.00e+00
knrm_llm_pretrain          0.56557        0.56557    +0.00e+00
knrm_name                  0.40506        0.40506    +0.00e+00
knrm_name_noaudit          0.56829        0.56829    +0.00e+00
knrm_name_v2               0.53078        0.53078    +0

## 5. С чего начинать разбор

Дальше — три готовых среза. Меняй `MODEL` и смотри.

In [7]:
# модели, обученные НА ИСПРАВЛЕННОЙ разметке; при их отсутствии — исходные
MODEL = "lgbm_cheap_audit" if "lgbm_cheap_audit" in models else "lgbm_cheap_v1"
SECOND_MODEL = "knrm_llm_audit" if "knrm_llm_audit" in models else "knrm_llm_pretrain"
print(f"основная {MODEL}, вторая {SECOND_MODEL}")

errors = df.with_columns((pl.col("target") - pl.col(MODEL)).abs().alias("ошибка"))

print(f"\n### {MODEL}: позитивы с самым низким скором (модель их пропустила)\n")
for row in errors.filter(pl.col("target") == 1).sort(MODEL).head(5).iter_rows(named=True):
    print(f"  скор {row[MODEL]:.4f} скор2 {row[SECOND_MODEL]:.4f} [{row['category']}]")
    print(f"    A: {row['name1'][:95]}")
    print(f"    Attr: {row['attributes1'][:95]}")
    print(f"    B: {row['name2'][:95]}")
    print(f"    Attr: {row['attributes2'][:95]}\n")

print(f"### {MODEL}: негативы с самым высоким скором (ложные срабатывания)\n")
for row in errors.filter(pl.col("target") == 0).sort(MODEL, descending=True).head(5).iter_rows(named=True):
    print(f"  скор {row[MODEL]:.4f} скор2 {row[SECOND_MODEL]:.4f} [{row['category']}]")
    print(f"    A: {row['name1'][:95]}")
    print(f"    Attr: {row['attributes1'][:95]}")
    print(f"    B: {row['name2'][:95]}")
    print(f"    Attr: {row['attributes2'][:95]}\n")


основная lgbm_cheap_audit, вторая knrm_llm_audit

### lgbm_cheap_audit: позитивы с самым низким скором (модель их пропустила)

  скор 0.0011 скор2 0.0203 [Аптека]
    A: очки с футляром-змейка melorsch мод. 2110 цвет 6 с линзами ecoplus 1.56 hmc -6.00 рц 60-62
    Attr: {"наличие флексов":"нет","затемнение":"без затемнения, с просветляющим покрытием.","бренд":"mel
    B: очки с футляром-змейка melorsch мод. 2110 цвет 6 с линзами ecoplus 1.56 hmc -8.00 рц 58-60
    Attr: {"диоптрии":"- 8.00","цвет оправы":"синий","назначение":"корректирующие","конструкция":"ободков

  скор 0.0018 скор2 0.0212 [Аптека]
    A: компьютерные очки для чтения с футляром-змейка dacchi мод. 32544 цвет 8 с линзами romeo 1.56 bl
    Attr: {"диоптрии":"+ 0.75","цвет оправы":"серебристый, черный","назначение":"для компьютера","констру
    B: очки для чтения с футляром-змейка dacchi мод. 32544 цвет 8 с линзами ecoplus 1.50 uc +0.75 рц 6
    Attr: {"наличие флексов":"да","затемнение":"без затемнения","бренд":"dacchi"

## 6. Доразметка: фиксация ошибок в метках

Журнал судейств — `members/darksteeld/data/label_audit.jsonl`. Каталог в
gitignore (`members/*/data/`), так что метки не уезжают в репозиторий. Файл
append-only: повторное судейство той же пары не затирает прежнее, при чтении
побеждает последнее.

**Ловушка, которую тут легко себе устроить.** Если размечать только худшие
ошибки модели и часть из них исправлять в её пользу, пересчёт PR-AUC на
исправленных метках вырастет **по построению**: отобраны ровно те пары, где
модель спорила с меткой, поэтому любое исправление двигает счёт в её сторону.
Число «сколько метрики съедает шум разметки», полученное так, не значит ничего
(CAMPAIGN_RULES #9 — адверсариальный проход, когда результат выглядит слишком
хорошо).

Поэтому у очереди три режима:

| режим | что показывает | для чего |
| --- | --- | --- |
| `fn` | позитивы с низким скором | найти пропуски и понять их природу |
| `fp` | негативы с высоким скором | найти ложные срабатывания |
| `random` | случайные пары, модель не участвует | **только отсюда** берётся доля шума и потолок метрики |

Первые два — инструмент поиска дефектов. Оценивать долю ошибок в разметке по
ним нельзя.

In [8]:
import datetime as _dt
import getpass
import json as _json

AUDIT_FILE = REPO / "members" / "darksteeld" / "data" / "label_audit.jsonl"
AUDIT_FILE.parent.mkdir(parents=True, exist_ok=True)
AUDITOR = getpass.getuser()


def _audit_log() -> dict:
    """Последнее судейство по каждой паре."""
    out = {}
    if AUDIT_FILE.is_file():
        for line in AUDIT_FILE.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = _json.loads(line)
                out[(r["id1"], r["id2"])] = r
    return out


def _pick(*candidates):
    for c in candidates:
        if c in models:
            return c
    return None


# Голосование ведут три НЕизбыточные модели: бустинг, нейросеть и обучаемая-без-
# меток текстовая похожесть. Нейросетевой голос теперь у knrm_joint: он читает и
# название, и атрибуты, то есть заменяет собой обе прежние однополевые сети —
# держать их обе значило бы считать нейросетевой голос дважды. Обученные на исправленной разметке предпочитаются
# своим исходным близнецам — держать в голосовании обе версии одной модели
# значит считать её голос дважды. const_prior, бленды и свип весов исключены.
CONSENSUS_MODELS = [m for m in (
    _pick("lgbm_cheap_audit", "lgbm_cheap_v1"),
    _pick("knrm_joint", "knrm_llm_audit", "knrm_llm_pretrain", "knrm_name_v2"),
    _pick("name_tfidf_cos"),
) if m]

# target с учётом уже сделанных исправлений: очереди строятся по нему, иначе
# отсуженные пары продолжали бы всплывать как «ошибки модели»
_fixes = {k: v["audited_label"] for k, v in _audit_log().items()
          if v["audited_label"] >= 0 and v["audited_label"] != v["original_target"]}
if _fixes:
    _corrected = [
        _fixes.get((a, b), t)
        for a, b, t in zip(df["id1"].to_list(), df["id2"].to_list(), df["target"].to_list())
    ]
    df = df.with_columns(pl.Series("target_audited", _corrected, dtype=pl.Int64))
else:
    df = df.with_columns(pl.col("target").alias("target_audited"))
TARGET = "target_audited"


# ---- in-fold предсказания -------------------------------------------------
# Модель, обученная на ВСЕХ парах, предсказывает их же. Ошибка на паре, которую
# модель видела в обучении и по которой её штрафовали, отсекает объяснение
# «модель не обобщила» и оставляет «метка спорит с остальными данными».
#
# Приём стоит ровно столько, сколько модель успевает запомнить обучающую
# выборку. Разрыв in-fold минус OOF печатается ниже: у бустинга на 21 признаке
# он мал (+0.036) и его in-fold очередь на 70% совпадает с OOF, у KNRM по
# атрибутам с таблицей на 650k строк он большой (+0.162) и совпадение лишь 42%.
# Поэтому по умолчанию in-fold режимы смотрят на KNRM, а не на бустинг.
INFOLD_DIR = REPO / "validation" / "predictions_v2_infold" / "darksteeld"
INFOLD_PREFIX = "infold_"

_infold_loaded = []
if INFOLD_DIR.is_dir():
    for _d in sorted(INFOLD_DIR.iterdir()):
        if not _d.is_dir() or not all((_d / f"{f}.csv").is_file() for f in FOLDS):
            continue
        _frame = pl.concat([pl.read_csv(_d / f"{f}.csv") for f in FOLDS])
        if not (_frame["id1"].equals(df["id1"]) and _frame["id2"].equals(df["id2"])):
            print(f"  {_d.name}: порядок пар отличается — пропущено")
            continue
        df = df.with_columns(_frame["predict"].alias(INFOLD_PREFIX + _d.name))
        _infold_loaded.append(INFOLD_PREFIX + _d.name)

# по какой модели строить in-fold очереди: та, что сильнее запоминает
INFOLD_MODEL = next((m for m in (INFOLD_PREFIX + "knrm_attrs_llm_full",
                                 INFOLD_PREFIX + "lgbm_cheap_audit") if m in _infold_loaded),
                    _infold_loaded[0] if _infold_loaded else None)


def infold_gap(verbose: bool = True) -> dict:
    """Насколько каждая модель подогналась под обучающие пары (in-fold AP минус OOF AP)."""
    gaps = {}
    y = df[TARGET].to_numpy().astype(float)
    for column in _infold_loaded:
        plain = column[len(INFOLD_PREFIX):]
        if plain not in models:
            continue
        gap = _average_precision(y, df[column].to_numpy()) - \
              _average_precision(y, df[plain].to_numpy())
        gaps[plain] = gap
        if verbose:
            note = "запоминает слабо — очередь будет как OOF" if gap < 0.05 else "запоминает"
            print(f"    {plain:<24} in-fold − OOF = {gap:+.4f}   {note}")
    return gaps


def _average_precision(target, score):
    import numpy as _np
    order = _np.argsort(-score, kind="mergesort")
    labels, ranked = target[order], score[order]
    cumulative = _np.cumsum(labels)
    if cumulative[-1] == 0:
        return 0.0
    last = _np.r_[ranked[1:] != ranked[:-1], True]
    precision = cumulative[last] / (_np.arange(len(labels))[last] + 1)
    recall = cumulative[last] / cumulative[-1]
    return float(_np.sum(_np.diff(_np.r_[0.0, recall]) * precision))


def _disagreement():
    """|target - средний скор моделей|: насколько дружно модели спорят с меткой."""
    mean_score = sum(pl.col(m) for m in CONSENSUS_MODELS) / len(CONSENSUS_MODELS)
    return (pl.col(TARGET) - mean_score).abs()


def audit_queue(mode: str = "fn", model: str = None, n: int = 30, seed: int = 0):
    """Очередь на просмотр; метки берутся уже с учётом сделанных исправлений.

    fn          позитивы с низким скором модели — пропуски
    fp          негативы с высоким скором модели — ложные срабатывания
    consensus   пары, где с меткой спорят ВСЕ модели голосования
    random      случайные пары; только отсюда берётся несмещённая доля шума

    in-fold режимы: модель видела эту пару в обучении и всё равно ошиблась —
    объяснение «не обобщила» отпадает, остаётся «метка спорит с данными»
    infold_fn   позитивы, которые модель не подтянула, хотя училась на них
    infold_fp   негативы, которые модель не отвергла, хотя училась на них
    infold_both обе in-fold модели ошибаются на паре одновременно
    """
    model = model or MODEL
    if mode.startswith("infold"):
        if INFOLD_MODEL is None:
            raise ValueError("нет in-fold предсказаний; собери их: "
                             "members/darksteeld/src/infold_predictions.py")
        model = model if model.startswith(INFOLD_PREFIX) else INFOLD_MODEL
    if mode == "fn":
        q = df.filter(pl.col(TARGET) == 1).sort(model).head(n)
    elif mode == "fp":
        q = df.filter(pl.col(TARGET) == 0).sort(model, descending=True).head(n)
    elif mode == "infold_fn":
        q = df.filter(pl.col(TARGET) == 1).sort(model).head(n)
    elif mode == "infold_fp":
        q = df.filter(pl.col(TARGET) == 0).sort(model, descending=True).head(n)
    elif mode == "infold_both":
        if len(_infold_loaded) < 2:
            raise ValueError("нужны in-fold предсказания минимум двух моделей")
        _err = sum((pl.col(TARGET) - pl.col(c)).abs() for c in _infold_loaded) / len(_infold_loaded)
        q = df.with_columns(_err.alias("_spor")).sort("_spor", descending=True).head(n)
    elif mode == "consensus":
        q = df.with_columns(_disagreement().alias("_spor")).sort("_spor", descending=True).head(n)
    elif mode == "random":
        q = df.sample(n=n, seed=seed, shuffle=True)
    else:
        raise ValueError("mode: 'fn', 'fp', 'infold_fn', 'infold_fp', 'infold_both', "
                         "'consensus' или 'random'")
    return q.with_row_index("i").with_columns(
        pl.lit(mode).alias("_mode"), pl.lit(model).alias("_model")
    )


def against_label(r: dict) -> int:
    """Сколько моделей голосования спорят с меткой на этой паре."""
    return sum(1 for m in CONSENSUS_MODELS if (r[m] >= 0.5) != (r[TARGET] == 1))


def audit_show(queue, i: int, attribute_chars: int = 400) -> None:
    """Пара из очереди: метка, скоры ВСЕХ моделей, оба названия и оба набора атрибутов."""
    r = queue.row(i, named=True)
    seen = _audit_log().get((r["id1"], r["id2"]))
    mark_note = f"   [уже судили: {seen['audited_label']}]" if seen else ""
    changed = "" if r[TARGET] == r["target"] else f" (исходная {r['target']})"
    print(f"[{i}] target={r[TARGET]}{changed}  {r['category']}  "
          f"id {r['id1']} <-> {r['id2']}{mark_note}")
    for m in models:
        agrees = (r[m] >= 0.5) == (r[TARGET] == 1)
        vote = "*" if m in CONSENSUS_MODELS else " "
        print(f"     {vote}{'  ' if agrees else '!!'} {m:<24}{r[m]:.4f}")
    print(f"      спорят с меткой: {against_label(r)} из {len(CONSENSUS_MODELS)} "
          f"(* — участники голосования)")
    for side in ("1", "2"):
        a = r[f"attributes{side}"]
        if len(a) > attribute_chars:
            a = a[:attribute_chars] + f"… (+{len(a) - attribute_chars})"
        print(f"  {side}: {r['name' + side]}")
        print(f"     {a}")


def mark(queue, i: int, label: int, note: str = "") -> None:
    """Записать судейство. label: 1 — совпадение, 0 — нет, -1 — не определить."""
    r = queue.row(i, named=True)
    record = {
        "id1": int(r["id1"]), "id2": int(r["id2"]), "category": r["category"],
        "original_target": int(r["target"]), "audited_label": int(label),
        "against_label": against_label(r),
        "note": note, "mode": r["_mode"], "model": r["_model"],
        "auditor": AUDITOR, "at": _dt.datetime.now().isoformat(timespec="seconds"),
    }
    with AUDIT_FILE.open("a", encoding="utf-8") as sink:
        sink.write(_json.dumps(record, ensure_ascii=False) + "\n")
    verdict = "подтверждено" if label == int(r["target"]) else (
        "НЕ ОПРЕДЕЛИТЬ" if label == -1 else f"ИСПРАВЛЕНО {r['target']} -> {label}")
    print(f"[{i}] {verdict}" + (f"   {note}" if note else ""))


# Модели, которые вообще не видят меток: для них «отстала» не имеет смысла.
LABEL_FREE = {"const_prior", "name_exact", "name_tfidf_cos", "attr_jaccard",
              "name_tfidf_attr_blend"}


def model_freshness(verbose: bool = True) -> dict:
    """Сколько исправлений появилось уже ПОСЛЕ того, как модель была обучена.

    Время обучения берём по mtime предсказаний — переобучение всегда их
    перезаписывает, так что этот штамп нельзя забыть обновить вручную.
    """
    log = _audit_log()
    behind_by = {}
    for m in CONSENSUS_MODELS:
        predictions = PREDICTIONS / m / "fold_01.csv"
        if not predictions.is_file():
            continue
        trained_at = _dt.datetime.fromtimestamp(
            predictions.stat().st_mtime).isoformat(timespec="seconds")
        if m in LABEL_FREE:
            behind_by[m] = None
            if verbose:
                print(f"    {m:<24} меток не использует — переобучать нечего")
            continue
        behind = sum(1 for v in log.values()
                     if v["at"] > trained_at and v["audited_label"] >= 0
                     and v["audited_label"] != v["original_target"])
        behind_by[m] = behind
        if verbose:
            state = "актуальна" if behind == 0 else f"ОТСТАЛА на {behind} исправлений"
            print(f"    {m:<24} обучена {trained_at}  {state}")
    return behind_by


print(f"журнал: {AUDIT_FILE.relative_to(REPO)}  ({len(_audit_log())} пар отсужено, "
      f"{len(_fixes)} исправлений применено к target_audited)")
print(f"голосуют: {', '.join(CONSENSUS_MODELS)}")
model_freshness()
if _infold_loaded:
    print(f"in-fold предсказаний загружено: {len(_infold_loaded)}; "
          f"очереди строятся по {INFOLD_MODEL}")
    infold_gap()
print("поиск ошибок разметки:  label_pairs(audit_queue('consensus', n=30))")
print("по in-fold ошибкам:     label_pairs(audit_queue('infold_fn', n=30))")


журнал: members/darksteeld/data/label_audit.jsonl  (119 пар отсужено, 97 исправлений применено к target_audited)
голосуют: lgbm_cheap_audit, knrm_joint, name_tfidf_cos
    lgbm_cheap_audit         обучена 2026-08-14T23:54:45  актуальна
    knrm_joint               обучена 2026-08-16T08:52:25  актуальна
    name_tfidf_cos           меток не использует — переобучать нечего
in-fold предсказаний загружено: 2; очереди строятся по infold_knrm_attrs_llm_full
    knrm_attrs_llm_full      in-fold − OOF = +0.1620   запоминает
    lgbm_cheap_audit         in-fold − OOF = +0.0362   запоминает слабо — очередь будет как OOF
поиск ошибок разметки:  label_pairs(audit_queue('consensus', n=30))
по in-fold ошибкам:     label_pairs(audit_queue('infold_fn', n=30))


## 7. In-fold: пары, которые модель не выучила, даже увидев

OOF-ошибка показывает, где модель не обобщается. Это смесь двух вещей: пара
трудная — или метка неверна. Разделить их помогает **in-fold** предсказание: скор
той же модели на паре, которую она **видела при обучении**.

Модель с 283M параметров подгоняет обучающую выборку почти произвольно. Если
после обучения НА этой паре она всё равно уверенно спорит с меткой, то дело
скорее в метке: выучить противоречие было некуда, кроме как проигнорировать его.
Это стандартный приём поиска шума в разметке, и он даёт очередь чище, чем
OOF-ошибки, где наверх всплывают просто редкие товары.

Для каждой пары берётся среднее по **трём** моделям фолдов, которые её видели
(модель фолда K видит все пары, кроме фолда K). Лосс — обычный BCE:
`-[y·log p + (1-y)·log(1-p)]`; чем он выше, тем громче модель спорит с меткой.

Столбцы: `<модель>_infold` — in-fold скор, `<модель>_infold_loss` — его лосс,
`<модель>_oof_loss` — лосс OOF-скора для сравнения.

In [9]:
INSAMPLE = REPO / "members" / "darksteeld" / "data" / "insample"


def _bce(target, probability, eps: float = 1e-6):
    """Полярный BCE. Клип нужен: log(0) сделал бы лосс бесконечным, а нам его сортировать."""
    p = np.clip(np.asarray(probability, dtype=np.float64), eps, 1 - eps)
    y = np.asarray(target, dtype=np.float64)
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))


def load_infold(model: str) -> pl.DataFrame | None:
    """Среднее in-fold предсказание: по трём моделям фолдов, видевшим пару.

    Раскладка на диске: ``<модель>/trained_without_<K>/<J>.csv`` — предсказание
    модели, обученной без фолда K, на фолде J. При J != K пара была в обучении,
    то есть скор in-sample. Метрикой такие числа быть не могут и потому лежат
    отдельно от OOF, чтобы ни один скоринг не подобрал их по ошибке.
    """
    root = INSAMPLE / model
    if not root.is_dir():
        return None
    frames = []
    for fold in FOLDS:
        seen_by = [k for k in FOLDS if k != fold]      # модели, видевшие этот фолд
        parts = []
        for held_out in seen_by:
            path = root / f"trained_without_{held_out}" / f"{fold}.csv"
            if path.is_file():
                parts.append(pl.read_csv(path))
        if not parts:
            return None
        base = parts[0].select("id1", "id2")
        mean = sum(p["predict"] for p in parts) / len(parts)
        frames.append(base.with_columns(mean.alias("infold")))
    return pl.concat(frames)


INFOLD_MODELS = []
for _model in sorted(p.name for p in INSAMPLE.iterdir()) if INSAMPLE.is_dir() else []:
    _frame = load_infold(_model)
    if _frame is None:
        continue
    # Порядок пар в дампах — канонический порядок фолда, тот же, что у целей;
    # но сверяем, а не предполагаем: тихо перепутанные строки дали бы
    # правдоподобную и полностью неверную очередь на разметку.
    _joined = df.select("id1", "id2").join(_frame, on=["id1", "id2"], how="left")
    assert _joined.height == df.height, f"{_model}: join размножил строки"
    assert _joined["infold"].null_count() == 0, f"{_model}: не для всех пар есть in-fold"
    df = df.with_columns(_joined["infold"].alias(f"{_model}_infold"))
    INFOLD_MODELS.append(_model)

for _model in INFOLD_MODELS:
    df = df.with_columns([
        pl.Series(f"{_model}_infold_loss",
                  _bce(df[TARGET], df[f"{_model}_infold"])),
        pl.Series(f"{_model}_oof_loss", _bce(df[TARGET], df[_model])),
    ])

INFOLD_MAIN = next((m for m in ("knrm_joint", "knrm_attrs_llm", "knrm_name_noaudit")
                    if m in INFOLD_MODELS), None)
print(f"in-fold есть у {len(INFOLD_MODELS)}: {', '.join(INFOLD_MODELS) or '—'}")
if INFOLD_MAIN:
    _l = df[f"{INFOLD_MAIN}_infold_loss"]
    print(f"основная для очереди: {INFOLD_MAIN}")
    print(f"  лосс in-fold: медиана {_l.median():.4f}, p99 {_l.quantile(0.99):.4f}, "
          f"max {_l.max():.4f}")
    print(f"  лосс OOF:     медиана {df[f'{INFOLD_MAIN}_oof_loss'].median():.4f}")
    _hard = (_l > 1.0).sum()
    print(f"  пар с in-fold лоссом > 1.0: {_hard:,} ({100 * _hard / df.height:.2f}%) "
          f"— кандидаты в ошибки разметки")


_audit_queue_oof = audit_queue          # исходная версия, чтобы не потерять


def audit_queue(mode: str = "fn", model: str = None, n: int = 30, seed: int = 0):
    """То же, что раньше, плюс два режима поверх in-fold лосса.

    infold      пары с самым высоким in-fold лоссом: модель их видела и всё
                равно спорит с меткой — самый чистый сигнал шума в разметке
    infold_gap  пары, где in-fold лосс высок, А OOF-лосс низок: модель обобщает
                на них верно, но на самой паре метку не приняла
    """
    if mode not in ("infold", "infold_gap"):
        return _audit_queue_oof(mode=mode, model=model, n=n, seed=seed)
    if not INFOLD_MAIN:
        raise SystemExit("нет in-fold предсказаний — положи их в members/darksteeld/data/insample")
    model = model or INFOLD_MAIN
    loss = pl.col(f"{model}_infold_loss")
    q = (df.sort(loss, descending=True).head(n) if mode == "infold" else
         df.with_columns((loss - pl.col(f"{model}_oof_loss")).alias("_gap"))
           .sort("_gap", descending=True).head(n))
    return q.with_row_index("i").with_columns(
        pl.lit(mode).alias("_mode"), pl.lit(model).alias("_model"))


_audit_show_oof = audit_show


def audit_show(queue, i: int, attribute_chars: int = 400) -> None:
    """Как раньше, но со строкой in-fold: скор и лосс рядом с OOF."""
    _audit_show_oof(queue, i, attribute_chars=attribute_chars)
    if not INFOLD_MODELS:
        return
    r = queue.row(i, named=True)
    print("      in-fold (модель видела эту пару при обучении):")
    for m in INFOLD_MODELS:
        print(f"        {m:<22} скор {r[f'{m}_infold']:.4f}  лосс {r[f'{m}_infold_loss']:.3f}"
              f"   (OOF лосс {r[f'{m}_oof_loss']:.3f})")


print("\nочереди: audit_queue('infold')      — модель видела пару и всё равно спорит")
print("         audit_queue('infold_gap')  — спорит на самой паре, но обобщает верно")

in-fold есть у 3: knrm_attrs_llm, knrm_joint, knrm_name_noaudit
основная для очереди: knrm_joint
  лосс in-fold: медиана 0.1859, p99 2.5035, max 5.9419
  лосс OOF:     медиана 0.2122
  пар с in-fold лоссом > 1.0: 37,928 (10.37%) — кандидаты в ошибки разметки

очереди: audit_queue('infold')      — модель видела пару и всё равно спорит
         audit_queue('infold_gap')  — спорит на самой паре, но обобщает верно


In [10]:
def audit_summary(verbose: bool = True):
    """Что уже отсужено и что из этого можно честно заключить."""
    log = list(_audit_log().values())
    if not log:
        print("журнал пуст"); return None
    frame = pl.DataFrame(log)
    changed = frame.filter(pl.col("audited_label") != pl.col("original_target"))
    unclear = frame.filter(pl.col("audited_label") == -1)

    print(f"отсужено {frame.height} пар: подтверждено "
          f"{frame.height - changed.height}, исправлено {changed.height - unclear.height}, "
          f"не определить {unclear.height}\n")
    print(frame.group_by("mode").agg(
        pl.len().alias("пар"),
        (pl.col("audited_label") != pl.col("original_target")).sum().alias("исправлено"),
    ).sort("mode"))

    rnd = frame.filter((pl.col("mode") == "random") & (pl.col("audited_label") >= 0))
    print()
    if rnd.height >= 30:
        wrong = int((rnd["audited_label"] != rnd["original_target"]).sum())
        rate = wrong / rnd.height
        half = 1.96 * (rate * (1 - rate) / rnd.height) ** 0.5
        print(f"НЕСМЕЩЁННАЯ доля ошибок разметки (только режим random, n={rnd.height}): "
              f"{rate:.1%} ± {half:.1%}")
    else:
        print(f"случайных судейств {rnd.height} — мало для оценки доли шума. "
              f"Нужно хотя бы 30-50: q = audit_queue('random', n=50)")
    if verbose and changed.height:
        print("\nисправленные пары:")
        for r in changed.head(15).iter_rows(named=True):
            print(f"  {r['original_target']} -> {r['audited_label']}  [{r['category']}]  "
                  f"{r['id1']}<->{r['id2']}  {r['note']}")
    return frame


def audit_impact():
    """PR-AUC на исправленных метках. Читать только вместе с предупреждением ниже."""
    log = _audit_log()
    fixes = {k: v["audited_label"] for k, v in log.items()
             if v["audited_label"] >= 0 and v["audited_label"] != v["original_target"]}
    if not fixes:
        print("исправлений пока нет"); return
    corrected = target.copy()
    keys = list(zip(df["id1"].to_list(), df["id2"].to_list()))
    touched = 0
    for position, key in enumerate(keys):
        if key in fixes:
            corrected[position] = fixes[key]; touched += 1

    biased = sum(1 for k, v in log.items()
                 if v["mode"] in ("fn", "fp") and v["audited_label"] >= 0
                 and v["audited_label"] != v["original_target"])
    print(f"исправлено {touched} из {len(target):,} пар "
          f"({biased} из них найдены просмотром ошибок модели)\n")
    print(f"{'эксперимент':<24}{'как есть':>10}{'исправл.':>10}{'дельта':>10}")
    for model in models:
        score = df[model].to_numpy()
        before = float(np.mean([average_precision(target[(df['fold'] == f).to_numpy()],
                                                  score[(df['fold'] == f).to_numpy()]) for f in FOLDS]))
        after = float(np.mean([average_precision(corrected[(df['fold'] == f).to_numpy()],
                                                 score[(df['fold'] == f).to_numpy()]) for f in FOLDS]))
        print(f"{model:<24}{before:10.5f}{after:10.5f}{after - before:+10.5f}")
    if biased:
        print(f"\n!! {biased} исправлений пришли из режимов fn/fp, то есть из пар, где модель\n"
              f"   спорила с меткой. Рост метрики выше частично создан этим отбором и НЕ является\n"
              f"   оценкой потолка. Для потолка нужна доля ошибок из audit_summary() по 'random'.")


audit_summary()

отсужено 119 пар: подтверждено 16, исправлено 97, не определить 6

shape: (2, 3)
┌──────┬─────┬────────────┐
│ mode ┆ пар ┆ исправлено │
│ ---  ┆ --- ┆ ---        │
│ str  ┆ u32 ┆ u32        │
╞══════╪═════╪════════════╡
│ fn   ┆ 55  ┆ 53         │
│ fp   ┆ 64  ┆ 50         │
└──────┴─────┴────────────┘

случайных судейств 0 — мало для оценки доли шума. Нужно хотя бы 30-50: q = audit_queue('random', n=50)

исправленные пары:
  0 -> 1  [Продукты питания]  171798771536<->111669197956  
  0 -> 1  [Канцелярские товары]  60129579350<->395137010891  
  0 -> 1  [Канцелярские товары]  816043845045<->180388644697  
  0 -> 1  [Хобби и творчество]  386547068603<->515396099825  
  0 -> -1  [Продукты питания]  300647817608<->352187456650  
  0 -> 1  [Музыкальные инструменты]  635655278862<->214748511306  
  0 -> 1  [Музыкальные инструменты]  549755904744<->137439074162  
  0 -> 1  [Канцелярские товары]  163208873648<->317827708054  
  0 -> -1  [Товары для животных]  395137122939<->85899383676  
  0

id1,id2,category,original_target,audited_label,note,mode,model,auditor,at,against_label
i64,i64,str,i64,i64,str,str,str,str,str,i64
171798771536,111669197956,"""Продукты питания""",0,1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T17:48:07""",null
60129579350,395137010891,"""Канцелярские товары""",0,1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T17:58:30""",7
575525682743,231928331947,"""Музыкальные инструменты""",0,0,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T17:58:45""",3
816043845045,180388644697,"""Канцелярские товары""",0,1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T17:59:05""",7
386547068603,515396099825,"""Хобби и творчество""",0,1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T18:00:59""",4
300647817608,352187456650,"""Продукты питания""",0,-1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T18:03:00""",4
635655278862,214748511306,"""Музыкальные инструменты""",0,1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T18:03:16""",3
549755904744,137439074162,"""Музыкальные инструменты""",0,1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T18:03:58""",3
163208873648,317827708054,"""Канцелярские товары""",0,1,"""""","""fp""","""lgbm_cheap_v1""","""dmitrijrudenko""","""2026-08-14T18:05:20""",6


In [11]:
# Мнение Opus 5 по каждой паре батча. Вердикт МОДЕЛИ НЕ ПОКАЗЫВАЕТСЯ: он пишется
# в кэш, а в панель попадает только разметка — те куски названия и атрибутов,
# которые этот вердикт определяют. Так подсказка направляет взгляд, но не
# подменяет решение разметчика готовым ответом.
import concurrent.futures as _fut
import html as _html
import os as _os

LLM_MODEL = "claude-opus-5"
LLM_EFFORT = "medium"          # low | medium | high — глубина рассуждения
LLM_WORKERS = 8                # параллельных запросов
LLM_INPUT_CHARS = 3000         # сколько символов атрибутов отдаём модели

HIGHLIGHT_FILE = REPO / "members" / "darksteeld" / "data" / "llm_highlights.jsonl"

_LLM_SYSTEM = (
    "Ты сверяешь карточки товаров маркетплейса. Тебе дают две карточки: название, "
    "категория и атрибуты. Реши, один и тот же это товар или разные товары.\n\n"
    "Один и тот же товар — это одна и та же вещь одного производителя, в одном "
    "исполнении: совпадают модель/артикул, объём или вес, размер, цвет, "
    "комплектность, количество в упаковке. Разные названия одного и того же "
    "товара — это совпадение. Разный объём, размер, цвет, количество в упаковке "
    "или другой артикул — это разные товары, даже если названия почти одинаковы.\n\n"
    "Затем выпиши подстроки, по которым виден вывод: артикулы, модели, объёмы, "
    "размеры, цвета, количества — то, что совпадает (если товар один) или "
    "расходится (если товары разные). Требования к подстрокам:\n"
    "  - копируй ТОЧНО, символ в символ, из присланного текста; не пересказывай;\n"
    "  - 1-5 слов каждая, не больше 6 подстрок на карточку;\n"
    "  - бери только решающие места, а не всё подряд;\n"
    "  - можно брать и из названия, и из атрибутов."
)

_LLM_SCHEMA = {
    "type": "object",
    "properties": {
        "verdict": {"type": "string", "enum": ["same", "different", "unclear"]},
        "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
        "reason": {"type": "string"},
        "spans1": {"type": "array", "items": {"type": "string"}},
        "spans2": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["verdict", "confidence", "reason", "spans1", "spans2"],
    "additionalProperties": False,
}


def _llm_client():
    """Клиент Anthropic; None, если нет ключа или пакета."""
    if not _os.environ.get("ANTHROPIC_API_KEY"):
        return None
    try:
        import anthropic
    except ImportError:
        return None
    return anthropic.Anthropic(max_retries=4, timeout=180.0)


def _highlight_cache() -> dict:
    """Последний ответ модели по каждой паре."""
    out = {}
    if HIGHLIGHT_FILE.is_file():
        for line in HIGHLIGHT_FILE.read_text(encoding="utf-8").splitlines():
            if line.strip():
                r = _json.loads(line)
                out[(r["id1"], r["id2"])] = r
    return out


def _llm_ask(client, r: dict) -> dict:
    """Один запрос к модели по одной паре."""
    def card(side: str) -> str:
        attributes = r[f"attributes{side}"] or ""
        if len(attributes) > LLM_INPUT_CHARS:
            attributes = attributes[:LLM_INPUT_CHARS] + " …(обрезано)"
        return f"название: {r['name' + side]}\nатрибуты: {attributes}"

    prompt = (f"категория: {r['category']}\n\n"
              f"=== карточка 1 ===\n{card('1')}\n\n"
              f"=== карточка 2 ===\n{card('2')}")
    message = client.beta.messages.create(
        model=LLM_MODEL,
        max_tokens=6000,
        betas=["server-side-fallback-2026-07-01"],
        fallbacks="default",
        system=_LLM_SYSTEM,
        output_config={
            "effort": LLM_EFFORT,
            "format": {"type": "json_schema", "schema": _LLM_SCHEMA},
        },
        messages=[{"role": "user", "content": prompt}],
    )
    if message.stop_reason == "refusal":
        raise RuntimeError("модель отказалась отвечать по этой паре")
    text = next(b.text for b in message.content if b.type == "text")
    answer = _json.loads(text)
    answer.update({
        "id1": int(r["id1"]), "id2": int(r["id2"]),
        "model": message.model, "effort": LLM_EFFORT,
        "at": _dt.datetime.now().isoformat(timespec="seconds"),
    })
    return answer


def llm_highlight(rows, workers: int = LLM_WORKERS, refresh: bool = False) -> dict:
    """Спросить Opus 5 по всем парам батча; вернуть кэш. Вердикт не печатается."""
    cache = _highlight_cache()
    todo = [r for r in rows if refresh or (r["id1"], r["id2"]) not in cache]
    if not todo:
        return cache
    client = _llm_client()
    if client is None:
        print("подсказка Opus 5 выключена: нет ANTHROPIC_API_KEY (или пакета anthropic) — "
              "панель работает без выделения")
        return cache

    print(f"спрашиваю {LLM_MODEL} по {len(todo)} парам…", end="", flush=True)
    done = errors = 0
    with HIGHLIGHT_FILE.open("a", encoding="utf-8") as sink:
        with _fut.ThreadPoolExecutor(max_workers=workers) as pool:
            futures = {pool.submit(_llm_ask, client, r): r for r in todo}
            for future in _fut.as_completed(futures):
                r = futures[future]
                try:
                    answer = future.result()
                except Exception as failure:            # noqa: BLE001 — батч не должен падать целиком
                    errors += 1
                    if errors <= 3:
                        print(f"\n  пара {r['id1']}<->{r['id2']}: {type(failure).__name__}: {failure}")
                    continue
                sink.write(_json.dumps(answer, ensure_ascii=False) + "\n")
                cache[(answer["id1"], answer["id2"])] = answer
                done += 1
                print("." if done % 5 else str(done), end="", flush=True)
    print(f"  готово: {done}" + (f", ошибок {errors}" if errors else ""))
    return cache


def _mark_spans(text: str, spans) -> str:
    """Экранированный HTML, где найденные подстроки обёрнуты в <b>."""
    text = text or ""
    hits = []
    lowered = text.lower()
    for span in spans or []:
        span = (span or "").strip()
        if len(span) < 2:
            continue
        needle = span.lower()
        start = lowered.find(needle)
        while start != -1 and len(hits) < 200:
            hits.append([start, start + len(needle)])
            start = lowered.find(needle, start + len(needle))
    if not hits:
        return _html.escape(text)
    hits.sort()
    merged = [hits[0]]
    for start, end in hits[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    pieces, position = [], 0
    for start, end in merged:
        pieces.append(_html.escape(text[position:start]))
        pieces.append("<b style='background:#ffe9a8;color:#000;border-radius:2px'>"
                      + _html.escape(text[start:end]) + "</b>")
        position = end
    pieces.append(_html.escape(text[position:]))
    return "".join(pieces)


def llm_vs_human(verbose: bool = True):
    """Согласие модели с разметчиком — ТОЛЬКО по уже отсуженным парам.

    Функция намеренно молчит про пары, которые ещё не размечены: иначе вердикт
    модели утёк бы разметчику до его собственного решения.
    """
    judged, hints = _audit_log(), _highlight_cache()
    verdict_to_label = {"same": 1, "different": 0, "unclear": -1}
    rows = [(v, hints[k]) for k, v in judged.items() if k in hints]
    if not rows:
        print("нет пар, которые и отсужены, и просмотрены моделью")
        return None
    table = pl.DataFrame([{
        "человек": human["audited_label"],
        "модель": verdict_to_label[hint["verdict"]],
        "уверенность": hint["confidence"],
        "исходная": human["original_target"],
    } for human, hint in rows])
    if verbose:
        decided = table.filter((pl.col("человек") >= 0) & (pl.col("модель") >= 0))
        agree = (decided["человек"] == decided["модель"]).sum()
        print(f"пар с обоими решениями: {len(decided)}, совпало {agree} "
              f"({agree / max(len(decided), 1):.0%})")
        print(decided.group_by("человек", "модель").len().sort("человек", "модель"))
    return table


print(f"подсказка Opus 5: кэш {HIGHLIGHT_FILE.relative_to(REPO)} "
      f"({len(_highlight_cache())} пар просмотрено)")
print(f"ключ ANTHROPIC_API_KEY: {'есть' if _os.environ.get('ANTHROPIC_API_KEY') else 'НЕТ — подсказка выключится'}")
print("вердикт модели в панель не выводится, только выделение решающих мест")


подсказка Opus 5: кэш members/darksteeld/data/llm_highlights.jsonl (0 пар просмотрено)
ключ ANTHROPIC_API_KEY: НЕТ — подсказка выключится
вердикт модели в панель не выводится, только выделение решающих мест


### Панель разметки

Кнопки вместо вызовов `mark(...)` руками. Показывает пару целиком, по нажатию
пишет судейство в журнал и сразу переходит к следующей.

```python
label_pairs(audit_queue("fp", n=30))        # ложные срабатывания
label_pairs(audit_queue("random", n=50))    # контрольная выборка для оценки шума
```

Перед показом батча каждую пару смотрит Opus 5. **Его вердикт в панель не
выводится** — видно только выделенные жирным куски названия и атрибутов, по
которым этот вердикт получен: артикулы, объёмы, размеры, цвета, количества.
Подсказка направляет взгляд на решающее место, а решение остаётся за
разметчиком. Отключить: `label_pairs(..., highlight=False)`.

Пары, уже отсуженные ранее, пропускаются автоматически — очередь можно
перезапускать, не боясь размечать одно и то же дважды (`skip_done=False`, если
нужно пересудить).


In [12]:
import ipywidgets as W
from IPython.display import display


def _pair_html(r: dict, model: str, hint: dict | None = None) -> str:
    spans1 = hint.get("spans1") if hint else None
    spans2 = hint.get("spans2") if hint else None

    def chip(m: str) -> str:
        agrees = (r[m] >= 0.5) == (r[TARGET] == 1)
        colour = "#0a7" if agrees else "#c33"
        weight = "700" if m == model else "400"
        border = ("2px solid #468" if m in CONSENSUS_MODELS else "1px solid #ddd")
        return (
            f"<span style='display:inline-block;margin:2px 4px 2px 0;padding:2px 7px;"
            f"border:{border};border-radius:10px;font-size:11px;font-weight:{weight}'>"
            f"{_html.escape(m)} <span style='color:{colour};font-weight:700'>{r[m]:.3f}</span></span>"
        )

    def block(side: str, spans) -> str:
        return (
            f"<div style='flex:1;min-width:0;padding:8px 10px;border:1px solid #ddd;border-radius:6px'>"
            f"<div style='font-weight:600;margin-bottom:4px'>"
            f"{_mark_spans(r['name' + side], spans)}</div>"
            f"<div style='font-size:11px;color:#666;max-height:190px;overflow:auto;"
            f"word-break:break-word'>{_mark_spans(r['attributes' + side], spans)}</div>"
            f"<div style='font-size:10px;color:#999;margin-top:4px'>id {r['id' + side]}</div></div>"
        )

    against = against_label(r)
    total = len(CONSENSUS_MODELS)
    verdict = (f"<span style='color:#c33;font-weight:700'>с меткой спорят {against} из {total}</span>"
               if against > total / 2
               else f"<span style='color:#666'>с меткой спорят {against} из {total}</span>")
    # про мнение Opus 5 сообщаем только факт, что оно есть; само решение скрыто
    hint_note = (" &nbsp;·&nbsp; <span style='color:#888'>жирным — что смотрел Opus 5</span>"
                 if hint else "")
    colour = "#0a7" if r[TARGET] == 1 else "#a33"
    fixed = "" if r[TARGET] == r["target"] else f" <span style='color:#888'>(исходная {r['target']})</span>"
    return (
        f"<div style='font-family:system-ui;font-size:12px'>"
        f"<div style='margin-bottom:6px'>"
        f"<span style='color:{colour};font-weight:700'>target = {r[TARGET]}</span>{fixed}"
        f" &nbsp;·&nbsp; {_html.escape(r['category'])} &nbsp;·&nbsp; {r['fold']}"
        f" &nbsp;·&nbsp; {verdict}{hint_note}</div>"
        f"<div style='margin-bottom:8px;line-height:1.9'>"
        f"{''.join(chip(m) for m in [model] + [x for x in models if x != model])}</div>"
        f"<div style='display:flex;gap:10px'>{block('1', spans1)}{block('2', spans2)}</div></div>"
    )


def label_pairs(queue, skip_done: bool = True, highlight: bool = True) -> None:
    """Панель ручной разметки: кнопки пишут судейство и переходят к следующей паре.

    highlight=True — перед показом спросить Opus 5 по всему батчу и выделить
    жирным места, определяющие вывод. Сам вывод модели не показывается.
    """
    done = _audit_log() if skip_done else {}
    rows = [r for r in queue.iter_rows(named=True) if (r["id1"], r["id2"]) not in done]
    if not rows:
        print("в этой очереди все пары уже отсужены — skip_done=False, чтобы пересудить")
        return
    hints = llm_highlight(rows) if highlight else {}
    model = rows[0]["_model"]
    state = {"position": 0, "marked": 0}

    progress = W.HTML()
    body = W.HTML()
    note = W.Text(placeholder="комментарий (необязательно)", layout=W.Layout(width="100%"))
    log_area = W.Output(layout=W.Layout(max_height="120px", overflow="auto"))

    def render() -> None:
        if state["position"] >= len(rows):
            progress.value = (f"<b>готово</b> — отсужено {state['marked']} из {len(rows)}. "
                              f"Дальше: <code>audit_summary()</code>")
            body.value = ""
            for widget in buttons:
                widget.disabled = True
            note.disabled = True
            return
        r = rows[state["position"]]
        progress.value = (f"<b>{state['position'] + 1} / {len(rows)}</b> &nbsp; "
                          f"режим <code>{r['_mode']}</code> &nbsp; отсужено {state['marked']}")
        body.value = _pair_html(r, model, hints.get((r["id1"], r["id2"])))

    def judge(label: int):
        def handler(_):
            r = rows[state["position"]]
            with log_area:
                mark(queue, int(r["i"]), label, note.value.strip())
            note.value = ""
            state["marked"] += 1
            state["position"] += 1
            render()
        return handler

    def skip(_):
        state["position"] += 1
        note.value = ""
        render()

    def back(_):
        state["position"] = max(0, state["position"] - 1)
        render()

    match = W.Button(description="Совпадение (1)", button_style="success",
                     layout=W.Layout(width="150px"))
    differ = W.Button(description="Не совпадает (0)", button_style="danger",
                      layout=W.Layout(width="150px"))
    unclear = W.Button(description="Не определить", button_style="warning",
                       layout=W.Layout(width="140px"))
    skip_button = W.Button(description="Пропустить", layout=W.Layout(width="110px"))
    back_button = W.Button(description="← Назад", layout=W.Layout(width="90px"))
    buttons = [match, differ, unclear, skip_button, back_button]

    match.on_click(judge(1))
    differ.on_click(judge(0))
    unclear.on_click(judge(-1))
    skip_button.on_click(skip)
    back_button.on_click(back)

    render()
    display(W.VBox([progress, body, note, W.HBox(buttons), log_area]))


print("панель готова:  label_pairs(audit_queue('consensus', n=30))")
print("без подсказки модели: label_pairs(..., highlight=False)")


панель готова:  label_pairs(audit_queue('consensus', n=30))
без подсказки модели: label_pairs(..., highlight=False)


In [17]:
label_pairs(audit_queue('infold_fn', n=30))

подсказка Opus 5 выключена: нет ANTHROPIC_API_KEY (или пакета anthropic) — панель работает без выделения


In [14]:
# PR-AUC по категориям для всех моделей сразу — где какая сильнее
rows = []
for category in sorted(df["category"].unique().to_list()):
    mask = (df["category"] == category).to_numpy()
    row = {"категория": category, "пар": int(mask.sum()),
           "prior": round(float(target[mask].mean()), 3)}
    for model in models:
        row[model] = round(average_precision(target[mask], df[model].to_numpy()[mask]), 4)
    rows.append(row)

per_category = pl.DataFrame(rows)
with pl.Config(tbl_rows=25, tbl_cols=20, fmt_str_lengths=30):
    print(per_category.sort(MODEL))

shape: (20, 28)
┌───────────┬───────┬───────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬───┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┬──────────┐
│ категория ┆ пар   ┆ prior ┆ attr_jac ┆ blend3_e ┆ blend3_o ┆ blend_lg ┆ blend_lg ┆ const_pr ┆ knrm_att ┆ … ┆ lgbm_che ┆ lgbm_che ┆ lgbm_knr ┆ lgbm_knr ┆ lgbm_llm ┆ name_exa ┆ name_tfi ┆ name_tfi ┆ stack3_l ┆ stack3_l │
│ ---       ┆ ---   ┆ ---   ┆ card     ┆ qual     ┆ pt       ┆ bm_knrm_ ┆ bm_knrm_ ┆ ior      ┆ rs       ┆   ┆ ap_audit ┆ ap_v1    ┆ m_insamp ┆ m_nested ┆ ---      ┆ ct       ┆ df_attr_ ┆ df_cos   ┆ ogit     ┆ ogit_ful │
│ str       ┆ i64   ┆ f64   ┆ ---      ┆ ---      ┆ ---      ┆ 50       ┆ audit_50 ┆ ---      ┆ ---      ┆   ┆ ---      ┆ ---      ┆ le       ┆ ---      ┆ f64      ┆ ---      ┆ blend    ┆ ---      ┆ ---      ┆ l        │
│           ┆       ┆       ┆ f64      ┆ f64      ┆ f64      ┆ ---      ┆ ---      ┆ f64      ┆ f64 

In [15]:
# Где модели расходятся сильнее всего — сырьё для бленда:
# пара, которую одна модель уверенно считает совпадением, а другая — нет.
A, B = "lgbm_cheap_v1", "knrm_llm_pretrain"
if A in models and B in models:
    print(f"корреляция {A} и {B}: "
          f"{np.corrcoef(df[A].to_numpy(), df[B].to_numpy())[0, 1]:.4f}\n")
    gap = df.with_columns((pl.col(A) - pl.col(B)).alias("разрыв"))
    for label, frame in ((f"{A} уверен, {B} нет", gap.sort("разрыв", descending=True)),
                         (f"{B} уверен, {A} нет", gap.sort("разрыв"))):
        print(f"### {label}")
        for row in frame.head(4).iter_rows(named=True):
            print(f"  target={row['target']:.0f}  {A} {row[A]:.3f}  {B} {row[B]:.3f}  [{row['category']}]")
            print(f"    A: {row['name1'][:90]}")
            print(f"    B: {row['name2'][:90]}")
        print()

корреляция lgbm_cheap_v1 и knrm_llm_pretrain: 0.6217

### lgbm_cheap_v1 уверен, knrm_llm_pretrain нет
  target=1  lgbm_cheap_v1 0.975  knrm_llm_pretrain 0.016  [Музыкальные инструменты]
    A: акустическая гитара naranda dg220cbk
    B: акустическая гитара naranda dg220cbk
  target=1  lgbm_cheap_v1 0.960  knrm_llm_pretrain 0.039  [Музыкальные инструменты]
    A: d012a гитара акустическая, doff
    B: d012a гитара акустическая, doff
  target=1  lgbm_cheap_v1 0.919  knrm_llm_pretrain 0.006  [Музыкальные инструменты]
    A: dg220bk акустическая гитара naranda
    B: dg220bk акустическая гитара naranda
  target=1  lgbm_cheap_v1 0.984  knrm_llm_pretrain 0.076  [Музыкальные инструменты]
    A: электрогитара schecter sgr banshee-6 wsn
    B: электрогитара schecter sgr banshee-6 wsn

### knrm_llm_pretrain уверен, lgbm_cheap_v1 нет
  target=0  lgbm_cheap_v1 0.019  knrm_llm_pretrain 0.990  [Обувь]
    A: туфли respect
    B: respect / туфли
  target=0  lgbm_cheap_v1 0.023  knrm_llm_pretrain 0.98

---

Таблица `df` дальше твоя. Полезные колонки, которых в ней ещё нет, но которые
легко добавить: разобранный JSON атрибутов (`json.loads` по `attributes1/2`),
длины названий, число общих токенов. Признаки, которые уже считает продовый
код, лучше брать оттуда, чтобы анализ и модель не разошлись:

```python
import sys; sys.path.insert(0, str(REPO / "members" / "darksteeld" / "src"))
from pair_features import build_features, FEATURE_NAMES
```